## URI Analysis Skript und Playground für die Distributionsanalysse

In diesem notebook werden einige Ideen getestet, um eine möglichst Aussagekräftige Analyse von URI-Fetch-Results zu ermöglichen.

Ziel ist es, zu Beginn erstmal grob die returnierten HTTP-Statuscodes zu analysieren, um zu schauen ob der Link überhaupt erreichbar ist.

Desweiteren wäre es von Nutzen eine Methode zu implementieren, die es ermöglicht folgende Dinge aus dem Result zu bekommen:

- Ist das Result ein direkter Dateidownload
- Wenn ja wie groß ist die Datei
- Von welchem Typ ist die Datei
- Einen kleinen Ausschnitt der Datei zwischenspeichern

- Wenn nein. Was ist in der Response? HTML, XML eines WMF Endpunkts, REST



Zum anderen soll in diesem Notebook geprüft werden wie man generell am besten die Distributionen überprüft

Was soll gemacht werden wenn es abartig viele Distributionen sind? Teilweise sind es über 50 50 bis sogar 100 Distributionen. wie soll sowas überprüft werden.

Wie gibt man bei einem solchen Fall Kontext für ein LLM-Agent mit?

Wie soll zwischen Geo und Non Geo Datensätzen unterschieden werden?
- Im Interview I2 haben wir erfahren dass die Distributionen eigentlich lediglich unterschiedliche "Distributionen" des gleichen Datensatzes sein sollten.
- Das Problem an der Sache ist bei Geodaten sind die Hauptdistributionen fast immer WMS WFS Endpunkte also mehr oder weniger nochmal ein Verweis auf Metadaten über die man dann auf die eigentlichen Daten kommt.
- Heisst Geodaten sind dann in diesem Punkt immer falsch? 
- Wie weit sollte man den ganzen Punkt Distributionen überhaupt mit in die Analyse packen?

In [11]:
# Wir fangen an mit ein paar einfachen URI-fetches und schauen uns mal an wie das so zurück kommt.


URI_CSV_DOWNLOAD = "https://offenedaten.aachen.de/dataset/6cbe0cbe-b1d6-4755-9e76-b1cec04bd857/resource/24b698d1-a72d-464d-8887-5cafec653ca8/download/2021_november.csv"

URI_GEO_JSON_DOWNLOAD = "https://geoserver.digitale-mrn.de/geoserver/metropolatlas/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=metropolatlas%3AMRN_LDI_PREASSURE_4326_Point&outputFormat=application%2Fjson"

URI_WMS_GET_CAPABILITIES = "https://geoserver.digitale-mrn.de/geoserver/metropolatlas/MRN_LDI_PREASSURE_4326_Point/wms?request=GetCapabilities"

URI_GPKG_DOWNLOAD = "https://geoserver.digitale-mrn.de/geoserver/metropolatlas/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=metropolatlas%3AMRN_LDI_PREASSURE_4326_Point&outputFormat=application%2Fx-gpkg"

URI_PDF_DOWNLOAD = "https://www.offenbach.de/medien/bindata/of/Statistik_und_wahlen_/dir-18/STAK-2024-Statistik-Aktuell-Nr.24-Zensus-2022-in-Offenbach-erste-Ergebnisse.pdf"

URI_XLSX_DOWNLOAD = "https://www.offenbach.de/medien/bindata/of/Statistik_und_wahlen_/dir-18/dir-38/20240625_Zensus-2022-Ergebnis-f.Offenbach-am-Main-Grundinfo-Bevoelkerung.xlsx"

URI_WFS_GET_CAPABILITIES = "https://geoportal.freiburg.de/wfs/gut_parken/gut_parken?REQUEST=GetCapabilities&SERVICE=WFS"

URI_500_ERROR_RESPONSE = "https://portal-chemnitz.opendata.arcgis.com/datasets/chemnitz::spielplaetzepunkte.zip?outSR=%7B%22latestWkid%22%253A3857%252C%22wkid%22%253A102100%7D"

URI_400_ERROR_RESPONSE = "https://hub.arcgis.com/api/download/v1/items/4c331993dab54b49bbc9debfc5928ec3/zip?redirect=true&layers=0"







def fetch_uri(uri):

    import requests

    try:
        response = requests.get(uri)
        print(f"URI: {uri}")
        print(f"Status Code: {response.status_code}")
        print(f"Content-Type: {response.headers.get('Content-Type')}")
        print("-" * 40)
    except Exception as e:
        print(f"Error fetching URI: {uri}")
        print(f"Error: {e}")
        print("-" * 40)



uris = [
    URI_CSV_DOWNLOAD,
    URI_GEO_JSON_DOWNLOAD,
    URI_WMS_GET_CAPABILITIES,
    URI_GPKG_DOWNLOAD,
    URI_PDF_DOWNLOAD,
    URI_XLSX_DOWNLOAD,
    URI_WFS_GET_CAPABILITIES,
    URI_500_ERROR_RESPONSE,
    URI_400_ERROR_RESPONSE
]


for uri in uris:
    fetch_uri(uri)

URI: https://offenedaten.aachen.de/dataset/6cbe0cbe-b1d6-4755-9e76-b1cec04bd857/resource/24b698d1-a72d-464d-8887-5cafec653ca8/download/2021_november.csv
Status Code: 200
Content-Type: text/csv
----------------------------------------
URI: https://geoserver.digitale-mrn.de/geoserver/metropolatlas/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=metropolatlas%3AMRN_LDI_PREASSURE_4326_Point&outputFormat=application%2Fjson
Status Code: 200
Content-Type: application/json;charset=UTF-8
----------------------------------------
URI: https://geoserver.digitale-mrn.de/geoserver/metropolatlas/MRN_LDI_PREASSURE_4326_Point/wms?request=GetCapabilities
Status Code: 200
Content-Type: text/xml
----------------------------------------
URI: https://geoserver.digitale-mrn.de/geoserver/metropolatlas/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=metropolatlas%3AMRN_LDI_PREASSURE_4326_Point&outputFormat=application%2Fx-gpkg
Status Code: 200
Content-Type: application/x-gpkg
--------------

Als nächstes probieren wir eine Lösung von github für die Ressourcetyp validierung für eine rdf File von govdata.de umzubauen.

In [ ]:
FILE_PATH = "../../data/sample_2026-05-21_08-57/geo_open-data-portal-rheinland-pfalz_02.rdf"

